In [58]:
from dotenv import load_dotenv

load_dotenv()

True

In [59]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate

In [60]:
loader = PyPDFLoader("../sample.pdf")
docs = loader.load()
len(docs)

1

In [61]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
text_chunks = splitter.split_documents(docs)
len(text_chunks)

2

In [62]:
embeddings = OllamaEmbeddings(model="qwen3-embedding")

In [63]:
vector_store = Chroma.from_documents(documents=docs, embedding=embeddings)

In [64]:
query = "What is Adobe Portable Document Format?"
result = vector_store.similarity_search(query=query)
result

[Document(id='584ddabe-ac38-4ff3-b1cb-9bfe5e8902f6', metadata={'moddate': '2013-10-28T15:24:13-04:00', 'producer': 'Acrobat Distiller 4.0 for Windows', 'total_pages': 1, 'page': 0, 'creator': 'Microsoft Word 8.0', 'author': 'cdaily', 'source': '../sample.pdf', 'title': 'This is a test PDF file', 'page_label': '1', 'creationdate': '2000-06-29T10:21:08+11:00'}, page_content="Adobe Acrobat PDF Files\nAdobe® Portable Document Format (PDF) is a universal file format that preserves all\nof the fonts, formatting, colours and graphics of any source document, regardless of\nthe application and platform used to create it.\nAdobe PDF is an ideal format for electronic document distribution as it overcomes the\nproblems commonly encountered with electronic file sharing.\n• Anyone, anywhere can open a PDF file. All you need is the free Adobe Acrobat\nReader. Recipients of other file formats sometimes can't open files because they\ndon't have the applications used to create the documents.\n• PDF file

In [65]:
context = ""
for data in result:
    context += data.page_content

context

"Adobe Acrobat PDF Files\nAdobe® Portable Document Format (PDF) is a universal file format that preserves all\nof the fonts, formatting, colours and graphics of any source document, regardless of\nthe application and platform used to create it.\nAdobe PDF is an ideal format for electronic document distribution as it overcomes the\nproblems commonly encountered with electronic file sharing.\n• Anyone, anywhere can open a PDF file. All you need is the free Adobe Acrobat\nReader. Recipients of other file formats sometimes can't open files because they\ndon't have the applications used to create the documents.\n• PDF files always print correctly on any printing device.\n• PDF files always display exactly as created, regardless of fonts, software, and\noperating systems. Fonts, and graphics are not lost due to platform, software, and\nversion incompatibilities.\n• The free Acrobat Reader is easy to download and can be freely distributed by\nanyone.\n• Compact PDF files are smaller than thei

In [66]:
llm = ChatOllama(model="minimax-m3:cloud")

In [53]:
# res = llm.invoke(
#     f"Can you provide me the answer based on provided context for my question,"
#     "context={context}, "
#     "question={query}"
# )

# print(res.content)

### Chain - context_generate | prompt | llm | strParser


In [ ]:
def get_context(query: str):
    data = vector_store.similarity_search(query=query)
    context = ""
    for doc in data:
        context += doc.page_content + "\n"

    return {"context": context, "question": query}

In [73]:
prompt = PromptTemplate.from_template("""
  You are a helpful assistant and provide answered based on the context for user question. ANd if you dont know the answer you can say "I don't know.
  Context: {context}
  Question: {question}
""")

In [74]:
rag_chain = get_context | prompt | llm

In [ ]:
res = rag_chain.invoke("?")
print(res.content)

Based on the context provided, **Adobe Acrobat PDF files** refers to files created using the **Adobe® Portable Document Format (PDF)**, which is a universal file format with the following key characteristics:

- **Preserves document integrity**: It preserves all the fonts, formatting, colours, and graphics of any source document, regardless of the application and platform used to create it.

- **Universal accessibility**: Anyone, anywhere can open a PDF file using the free Adobe Acrobat Reader, making it ideal for electronic document distribution since recipients don't need the original applications used to create the document.

- **Cross-platform compatibility**: PDF files always display exactly as created, regardless of fonts, software, and operating systems, with no loss of fonts or graphics due to platform, software, or version incompatibilities.

- **Reliable printing**: PDF files always print correctly on any printing device.

- **Free distribution**: The Acrobat Reader is free t